In [ ]:
import os
import pandas as pd
from urllib.parse import quote_plus
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()  # Automatically loads variables from local private .env file

DB_USER = os.getenv("DB_USER", "root")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_NAME = os.getenv("DB_NAME", "olist_dw")

engine = create_engine(f"mysql+pymysql://{DB_USER}:{quote_plus(DB_PASS)}@{DB_HOST}/{DB_NAME}")

# 1. Load Marketing Qualified Leads (MQL)
mql_path = os.path.join(DATA_DIR, "olist_marketing_qualified_leads_dataset.csv")
df_mql = pd.read_csv(mql_path)
df_mql["first_contact_date"] = pd.to_datetime(df_mql["first_contact_date"])
df_mql["origin"] = df_mql["origin"].fillna("unknown").str.strip()

# 2. Load Closed Deals
deals_path = os.path.join(DATA_DIR, "olist_closed_deals_dataset.csv")
df_deals = pd.read_csv(deals_path)
df_deals["won_date"] = pd.to_datetime(df_deals["won_date"])
df_deals["business_segment"] = df_deals["business_segment"].fillna("unknown").str.strip()
df_deals["lead_type"] = df_deals["lead_type"].fillna("unknown").str.strip()

# Write to MySQL
df_mql.to_sql("olist_mql", con=engine, if_exists="replace", index=False)
df_deals.to_sql("olist_closed_deals", con=engine, if_exists="replace", index=False)

# Add Primary & Foreign Key Indexes
with engine.begin() as conn:
    conn.execute(text("ALTER TABLE olist_mql MODIFY mql_id VARCHAR(50) NOT NULL, ADD PRIMARY KEY (mql_id);"))
    conn.execute(text("ALTER TABLE olist_closed_deals MODIFY mql_id VARCHAR(50) NOT NULL, ADD PRIMARY KEY (mql_id);"))
    conn.execute(text("ALTER TABLE olist_closed_deals MODIFY seller_id VARCHAR(50), ADD INDEX idx_deals_seller (seller_id);"))

print(f"Loaded {len(df_mql):,} MQLs and {len(df_deals):,} Closed Deals into {DB_NAME}!")

Loaded 8,000 MQLs and 842 Closed Deals into olist_dw!
